# Baseline Evaluation: Qwen2.5-7B-Instruct vs Qwen2.5-0.5B-Instruct on CNN/DailyMail

**Goal:** Establish baseline ROUGE for the teacher/student pair before distillation and fine-tuning experiments.

**Models:**
- Teacher: `Qwen/Qwen2.5-7B-Instruct`
- Student: `Qwen/Qwen2.5-0.5B-Instruct`

**Metrics:** ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-Lsum + summary length stats + a 'clean prediction' rate (preamble stripped).

**Hardware:** Single A100 80GB. bf16, batched generation, models loaded sequentially.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.46.0 datasets==2.21.0 evaluate==0.4.3 rouge_score==0.1.2 accelerate==1.0.1 sentencepiece

## 2. Imports & config

In [ ]:
import torch
import gc
import json
import re
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CONFIG = {
    'num_eval_samples': 1000,        # match your BART run for comparability
    'teacher_batch_size': 4,         # 7B in bf16 needs smaller batch
    'student_batch_size': 16,        # 0.5B can handle much more
    'max_input_tokens': 3000,        # leaves headroom under Qwen's 32k context
    'max_new_tokens': 160,           # target summary length
    'temperature': 0.0,              # greedy for reproducibility (deterministic baseline)
    'do_sample': False,
    'seed': 42,
    'results_dir': './baseline_results',
}
Path(CONFIG['results_dir']).mkdir(exist_ok=True)
torch.manual_seed(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))

## 3. Load CNN/DailyMail test split

In [ ]:
dataset = load_dataset('cnn_dailymail', '3.0.0', split='test')
print(f'Total test examples: {len(dataset)}')

if CONFIG['num_eval_samples'] is not None:
    dataset = dataset.shuffle(seed=CONFIG['seed']).select(range(CONFIG['num_eval_samples']))
    print(f'Using subset: {len(dataset)} examples')

ex = dataset[0]
print('\n--- Example article (first 400 chars) ---')
print(ex['article'][:400], '...')
print('\n--- Reference summary ---')
print(ex['highlights'])

## 4. Prompt template & post-processing

Qwen is a chat model so we apply its chat template. We also strip common preambles ("Here is a summary:", "Summary:", etc.) because they hurt ROUGE without reflecting actual quality.

In [ ]:
SYSTEM_PROMPT = (
    'You are a concise news summarizer. Write a short summary of the article in 2-3 sentences. '
    'Output only the summary itself, with no preamble, headers, or commentary.'
)

USER_TEMPLATE = 'Article:\n{article}\n\nSummary:'

PREAMBLE_RE = re.compile(
    r'^\s*(here(?:\s+is|\'s)?\s+(?:a\s+)?(?:brief\s+|short\s+|concise\s+)?summary[:\s\-]*|'
    r'summary[:\s\-]+|'
    r'the\s+article\s+(?:is\s+about|discusses|describes)[:\s\-]*)',
    re.IGNORECASE,
)

def clean_prediction(text: str) -> tuple[str, bool]:
    '''Strip leading preamble. Returns (cleaned_text, had_preamble).'''
    original = text
    stripped = PREAMBLE_RE.sub('', text).strip()
    # If preamble was on its own line, also strip a leading newline
    stripped = stripped.lstrip('\n').strip()
    return stripped, (stripped != original.strip())

# quick sanity check
for t in [
    'Here is a summary: The plane crashed in Texas.',
    'Summary: Officials confirmed three injuries.',
    'The plane crashed in Texas.',
]:
    print(repr(t), '→', clean_prediction(t))

## 5. ROUGE metric

In [ ]:
rouge = evaluate.load('rouge')
print('ROUGE metric loaded.')

## 6. Evaluation function

In [ ]:
def evaluate_model(model_name: str, dataset, config: dict, batch_size: int) -> dict:
    print(f'\n{"="*60}\nEvaluating: {model_name}\n{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # left padding needed for batched generation with causal LMs
    tokenizer.padding_side = 'left'

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE,
    )
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {n_params/1e9:.2f}B  | batch_size={batch_size}')

    articles = dataset['article']
    refs = dataset['highlights']
    raw_predictions, cleaned_predictions, had_preamble_flags = [], [], []
    start = time.time()

    with torch.no_grad():
        for i in range(0, len(articles), batch_size):
            batch_articles = articles[i:i+batch_size]

            # build chat-formatted prompts; truncate the article text itself if needed
            prompts = []
            for art in batch_articles:
                # quick char-level pre-truncation; tokenizer truncation handles the rest
                art_trunc = art[:12000]
                messages = [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': USER_TEMPLATE.format(article=art_trunc)},
                ]
                prompts.append(tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True))

            inputs = tokenizer(
                prompts,
                max_length=config['max_input_tokens'],
                truncation=True,
                padding=True,
                return_tensors='pt',
            ).to(DEVICE)

            output_ids = model.generate(
                **inputs,
                max_new_tokens=config['max_new_tokens'],
                do_sample=config['do_sample'],
                temperature=config['temperature'] if config['do_sample'] else 1.0,
                pad_token_id=tokenizer.pad_token_id,
            )
            # strip the prompt tokens; decode only newly generated tokens
            gen_ids = output_ids[:, inputs['input_ids'].shape[1]:]
            decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

            for raw in decoded:
                cleaned, had = clean_prediction(raw)
                raw_predictions.append(raw.strip())
                cleaned_predictions.append(cleaned)
                had_preamble_flags.append(had)

            if (i // batch_size) % 5 == 0:
                elapsed = time.time() - start
                done = i + len(batch_articles)
                rate = done / elapsed if elapsed > 0 else 0
                eta = (len(articles) - done) / rate if rate > 0 else 0
                print(f'  [{done}/{len(articles)}] elapsed={elapsed:.0f}s rate={rate:.2f}/s eta={eta:.0f}s')

    total_time = time.time() - start
    print(f'\nGeneration finished in {total_time:.1f}s ({len(articles)/total_time:.2f} samples/s)')

    # ROUGE on cleaned predictions (fairer to LLM outputs)
    scores_clean = rouge.compute(predictions=cleaned_predictions, references=refs, use_stemmer=True)
    scores_raw = rouge.compute(predictions=raw_predictions, references=refs, use_stemmer=True)

    avg_pred_len = sum(len(p.split()) for p in cleaned_predictions) / len(cleaned_predictions)
    avg_ref_len = sum(len(r.split()) for r in refs) / len(refs)
    preamble_rate = sum(had_preamble_flags) / len(had_preamble_flags)

    results = {
        'model': model_name,
        'num_samples': len(articles),
        'num_params_B': round(n_params/1e9, 3),
        'gen_time_s': round(total_time, 1),
        'samples_per_sec': round(len(articles)/total_time, 3),
        'rouge1_clean': round(scores_clean['rouge1'] * 100, 3),
        'rouge2_clean': round(scores_clean['rouge2'] * 100, 3),
        'rougeL_clean': round(scores_clean['rougeL'] * 100, 3),
        'rougeLsum_clean': round(scores_clean['rougeLsum'] * 100, 3),
        'rouge1_raw': round(scores_raw['rouge1'] * 100, 3),
        'rouge2_raw': round(scores_raw['rouge2'] * 100, 3),
        'rougeL_raw': round(scores_raw['rougeL'] * 100, 3),
        'avg_pred_words': round(avg_pred_len, 1),
        'avg_ref_words': round(avg_ref_len, 1),
        'preamble_rate': round(preamble_rate, 3),
    }
    print('\nResults:')
    for k, v in results.items():
        if k != 'model':
            print(f'  {k}: {v}')

    # save sample predictions for qualitative review
    safe_name = model_name.replace('/', '_')
    out_path = Path(config['results_dir']) / f'predictions_{safe_name}.json'
    with open(out_path, 'w') as f:
        json.dump({
            'config': config,
            'results': results,
            'samples': [
                {'reference': refs[i], 'prediction_raw': raw_predictions[i], 'prediction_clean': cleaned_predictions[i]}
                for i in range(min(20, len(refs)))
            ],
        }, f, indent=2)
    print(f'Saved samples → {out_path}')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return results

## 7. Run evaluation

Teacher first (slower per sample, needs smaller batch), then student. One model in VRAM at a time.

In [ ]:
all_results = []

all_results.append(evaluate_model(
    'Qwen/Qwen2.5-7B-Instruct', dataset, CONFIG, batch_size=CONFIG['teacher_batch_size']
))

all_results.append(evaluate_model(
    'Qwen/Qwen2.5-0.5B-Instruct', dataset, CONFIG, batch_size=CONFIG['student_batch_size']
))

## 8. Summary table

In [ ]:
import pandas as pd
df = pd.DataFrame(all_results)
cols = ['model', 'num_params_B', 'samples_per_sec',
        'rouge1_clean', 'rouge2_clean', 'rougeL_clean', 'rougeLsum_clean',
        'avg_pred_words', 'preamble_rate']
print(df[cols].to_string(index=False))

out_csv = Path(CONFIG['results_dir']) / 'baseline_rouge_qwen.csv'
out_json = Path(CONFIG['results_dir']) / 'baseline_rouge_qwen.json'
df.to_csv(out_csv, index=False)
with open(out_json, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved:\n  {out_csv}\n  {out_json}')

## What to expect & how to read it

- **Teacher (7B):** likely ROUGE-1 around 30–36 zero-shot. *Lower than fine-tuned BART-large-cnn* — that's normal. Instruction-tuned LLMs are penalized by ROUGE because they paraphrase rather than copy spans like extractive/fine-tuned models do. Don't panic.
- **Student (0.5B):** likely ROUGE-1 around 22–28. Smaller gap to teacher than the BART pair, but the *style gap* will be bigger — student tends to be wordier, less focused. Look at the saved sample predictions to see qualitative differences.
- **`preamble_rate`:** if either model has >0.3, the system prompt isn't being followed reliably. We can tighten the prompt in Phase 2.
- **`rouge_clean` vs `rouge_raw`:** the delta tells you how much preamble stripping helps. Report `clean` numbers as primary, `raw` as a note.

## Next steps
Once you share these numbers I'll set up Phase 2:
1. Teacher generates summaries on a training split (one-time cost, ~2–3hr for 50k examples)
2. Student trained via sequence-level KD on those generations + via direct SFT on gold for comparison
3. Both with and without LoRA — four configurations total
4. Phase 3 — same models, evaluated on XSum and SAMSum for cross-domain generalization